In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# ==========================================
# 1. Automatic Environment Detection
# ==========================================
try:
    # --- Colab Environment ---
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("☁️ Colab Environment Detected.")
except ImportError:
    # --- Local Environment (VS Code) ---
    from dotenv import load_dotenv
    current_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(current_dir, "..", ".."))
    env_path = os.path.join(project_root, ".env")
    load_dotenv(dotenv_path=env_path, override=True)
    print("💻 Local Environment Detected.")

# ==========================================
# 2. Initialize the Model
# ==========================================
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.1
)
print("🚀 Gemini model is ready!")

💻 Local Environment Detected.
🚀 Gemini model is ready!


In [2]:
# ==========================================
# Basic Concept of RunnablePassthrough
# ==========================================

# Scenario: We receive a number. We want to keep the original number AND multiply it by 2.
chain = RunnableParallel({
    "original": RunnablePassthrough(), # Pass the input without any changes
    "modified": lambda x: x * 2        # Modify the input
})

print("⏳ Executing basic passthrough chain...\n")
result = chain.invoke(10)

print(result)

⏳ Executing basic passthrough chain...

{'original': 10, 'modified': 20}


In [8]:
# ==========================================
# Practical Example: A Mini-RAG Simulator
# ==========================================

# 1. Create a fake database retrieval function
def get_country_info(country_name):
    fake_db = {
        "فرانسه": "پایتخت فرانسه پاریس است و به برج ایفل مشهور است.",
        "ژاپن": "پایتخت ژاپن توکیو است و سوشی غذای معروف آن است."
    }
    return fake_db.get(country_name, "اطلاعاتی یافت نشد.")

# 2. Define the Prompt (Expecting two variables: {context} and {question})
template = """
با توجه به اطلاعات زیر به سوال کاربر پاسخ بده:
اطلاعات: {context}

سوال کاربر: پایتخت {question} کجاست؟
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Build the RAG Chain (The Magic of LCEL)
# Notice how a dictionary behaves exactly like RunnableParallel here!
rag_chain = (
    {
        "context": get_country_info, 
        "question": RunnablePassthrough() # Passes "فرانسه" untouched to the prompt
    } 
    | prompt 
    | model 
    | StrOutputParser()
)

print("⏳ Executing RAG simulator chain...\n")
result = rag_chain.invoke("فرانسه")  # Input is the country name
print(result)

⏳ Executing RAG simulator chain...

پایتخت فرانسه پاریس است.


In [4]:
# ==========================================
# The Powerful .assign() Method
# ==========================================

# Input will be: {"num": 5}
# We want output to be: {"num": 5, "mult": 50}

# .assign() keeps the existing dictionary and appends a new key-value pair to it
chain = RunnablePassthrough.assign(mult=lambda x: x["num"] * 10)

print("⏳ Executing .assign() method chain...\n")
result = chain.invoke({"num": 5})

print(result)

⏳ Executing .assign() method chain...

{'num': 5, 'mult': 50}
